# Bayesian Optimization Logic (Using Real Dataset)
This notebook demonstrates the exact Machine Learning pipeline used in the Bayesian Optimization process, using the real dataset `WS2_ThermalCVD_BlankCells_17points.xlsx`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

np.random.seed(42)


## 1. Loading and Preprocessing the Dataset
We load the Excel dataset, filter for Thermal CVD experiments, handle missing values, and scale the data.


In [ ]:
# Load the actual dataset
df_raw = pd.read_excel('WS2_ThermalCVD_BlankCells_17points.xlsx')

# Filter for Thermal CVD
if 'TOCVD' in df_raw.columns:
    df = df_raw[df_raw['TOCVD'] == 'Thermal CVD'].copy().reset_index(drop=True)
else:
    df = df_raw.copy()
    
# Replace 'NS' (Not Specified) with NaN
df = df.replace('NS', np.nan)

# Define variables and their bounds
VARIABLES = {
    'GTE':      {'col': 'GTE',      'min': 500,  'max': 1100},
    'GTI':      {'col': 'GTI',      'min': 5,    'max': 60},
    'FRA':      {'col': 'FRA',      'min': 0,    'max': 600},
    'Pressure': {'col': 'Pressure', 'min': 1,    'max': 760},
}

CATEGORICAL_CONSTANTS = ['P1', 'P2', 'Substrate', 'CG', 'COM', 'PC', 'SA', 'Class']

# Label Encode Categoricals
for col in CATEGORICAL_CONSTANTS:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
        le = LabelEncoder()
        df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))

# Extract Numerical Variables
var_cols = [v['col'] for v in VARIABLES.values()]
X_var = df[var_cols].copy()

# Impute missing numerical values using mean strategy
imputer = SimpleImputer(strategy='mean')
X_var_imputed = imputer.fit_transform(X_var)

# Extract Encoded Categorical Constants
cat_cols = [f'{col}_enc' for col in CATEGORICAL_CONSTANTS if col in df.columns]
X_cat = df[cat_cols].values

# Combine Categoricals and Variables for the final Training Data
if X_cat.shape[1] > 0:
    X_train_raw = np.hstack([X_cat, X_var_imputed])
else:
    X_train_raw = X_var_imputed

# Extract Target (PL FWHM)
y_train_raw = df['PL FWHM'].values

# Scale the Inputs and Outputs
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train_raw)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_raw.reshape(-1, 1)).flatten()

print(f"Data loaded and preprocessed. Training shape: {X_train_scaled.shape}")


## 2. Training the Gaussian Process Surrogate Model
We use the Matern Kernel combined with the WhiteKernel to model the physical process while accounting for experimental noise.


In [ ]:
# Define the Kernel
kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5) \
       + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))

# Initialize and Fit the GP Regressor
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10, 
    normalize_y=False # Data is already scaled
)

gp.fit(X_train_scaled, y_train_scaled)

print("Optimized Kernel:", gp.kernel_)


## 3. Defining the Expected Improvement (EI) Acquisition Function
This function balances exploration (high uncertainty) and exploitation (low predicted FWHM) to find the next best experimental parameters.


In [ ]:
def expected_improvement(X_candidates, gp_model, y_best):
    """
    Computes the Expected Improvement at candidate points X_candidates.
    """
    mu, sigma = gp_model.predict(X_candidates, return_std=True)
    mu, sigma = mu.reshape(-1, 1), sigma.reshape(-1, 1)
    
    # Numerically safe implementation to avoid divide-by-zero warnings
    sigma = np.maximum(sigma, 1e-9)
    
    imp = y_best - mu
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0
        
    return ei.flatten()


## 4. Generating Candidates in the Search Space
We generate random candidate points strictly within the variable bounds, fix the categorical constants to their most frequent values, and evaluate them all using Expected Improvement.


In [ ]:
N_CANDIDATES = 5000

# Sample random continuous candidates within variable bounds
candidates_var = np.column_stack([
    np.random.uniform(info['min'], info['max'], N_CANDIDATES)
    for info in VARIABLES.values()
])

# Fix categorical constants at their most frequent (mode) encoded values
if len(cat_cols) > 0:
    const_fixed = np.array([df[col].mode()[0] for col in cat_cols])
    candidates_const = np.tile(const_fixed, (N_CANDIDATES, 1))
    
    # Combine constants and variables
    candidates_raw = np.hstack([candidates_const, candidates_var])
else:
    candidates_raw = candidates_var

# Scale the candidates
candidates_scaled = scaler_X.transform(candidates_raw)

# Get the best observed scaled FWHM
y_best_scaled = y_train_scaled.min()

# Compute Expected Improvement for all candidates
ei_values = expected_improvement(candidates_scaled, gp, y_best_scaled)

# Find the next experiment
best_idx = np.argmax(ei_values)
next_exp_var = candidates_var[best_idx]
max_ei = ei_values[best_idx]

print(f"Highest Expected Improvement: {max_ei:.4f}\n")
print("🎯 Next Suggested Optimization Variables:")
for i, name in enumerate(VARIABLES.keys()):
    print(f"{name:10s}: {next_exp_var[i]:.2f}")
